In [85]:
import pandas as pd
import numpy as np 
import re

# Load the dataset

In [86]:
GroupA = pd.read_csv(r"../data/raw/sales_group1_storesA.csv")
GroupB = pd.read_csv(r"../data/raw/sales_group2_storesB.csv")
GroupC = pd.read_csv(r"../data/raw/sales_group3_storesC.csv")

In [160]:
Stores = pd.read_csv(r"..\data\raw\stores.csv")
Category = pd.read_csv(r"..\data\raw\categories.csv")
Employee = pd.read_csv(r"..\data\raw\employees.csv")

In [87]:
GroupA

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method
0,1,3,21,3878,99,2,20/01/2025,184.16,8.1,Cash
1,7,3,88,465,997,1,20/10/2024,108.23,6.9,Cash
2,8,2,32,3997,131,1,07/12/2024,298.25,6.4,Debit Card
3,12,2,13,4985,151,2,26/07/2025,140.55,5.4,Cash
4,13,1,3,3210,205,4,09/03/2025,179.13,32.4,Credit Card
...,...,...,...,...,...,...,...,...,...,...
3247,9880,3,88,2625,1110,1,12/12/2024,150.49,8.9,Online
3248,9890,1,118,4693,1299,1,23/06/2025,180.56,25.4,Debit Card
3249,9894,3,21,2189,979,4,19/06/2025,245.33,4.6,Cash
3250,9895,1,82,261,1341,2,10/10/2024,197.45,22.0,Debit Card


# Combine all three sales-group files into a single table. Resolve renamed or reordered columns such as qty versus quantity and amount versus unit_price.

In [88]:
GroupA.columns

Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'quantity', 'sale_date', 'unit_price', 'discount_pct',
       'payment_method'],
      dtype='object')

In [89]:
GroupB = GroupB.rename(columns={'qty': 'quantity', 'amount': 'unit_price'}) 

In [90]:
GroupB.columns

Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'quantity', 'sale_date', 'unit_price', 'discount_pct',
       'payment_method'],
      dtype='object')

In [91]:
GroupC.columns

Index(['sale_id', 'store_id', 'employee_id', 'product_id', 'customer_id',
       'quantity', 'sale_date', 'unit_price', 'discount_pct', 'payment_method',
       'notes'],
      dtype='object')

In [92]:
Combine_Sales = pd.concat([GroupA, GroupB, GroupC], ignore_index=True)

In [93]:
Combine_Sales

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes
0,1,3,21,3878,99,2,20/01/2025,184.16,8.1,Cash,NaN
1,7,3,88,465,997,1,20/10/2024,108.23,6.9,Cash,NaN
2,8,2,32,3997,131,1,07/12/2024,298.25,6.4,Debit Card,NaN
3,12,2,13,4985,151,2,26/07/2025,140.55,5.4,Cash,NaN
4,13,1,3,3210,205,4,09/03/2025,179.13,32.4,Credit Card,NaN
...,...,...,...,...,...,...,...,...,...,...,...
9945,7937,10,87,4504,372,3,2025-04-16,109.51,33.7,Cash,NaN
9946,6711,11,63,2028,805,1,2025-10-16,51.18,26.7,Credit Card,Loyalty tier discount applied
9947,8450,9,27,26,288,1,2025-04-10,48.36,5.2,Credit Card,Loyalty tier discount applied
9948,637,10,39,2571,282,1,2025-11-23,115.75,28.0,Credit Card,Promo code WELCOME15 applied at checkout


# Format sale_date as DD/MM/YYYY.

In [ ]:
Combine_Sales['sale_date'] = pd.to_datetime(Combine_Sales['sale_date'],format='mixed')

In [162]:
Combine_Sales['sale_date'] = pd.to_datetime(
    Combine_Sales['sale_date'],
    format='%d/%m/%Y'
)

# Standardize text prices like "$123.45".

In [96]:
Combine_Sales['unit_price'].head(10)

0    184.16
1    108.23
2    298.25
3    140.55
4    179.13
5    126.69
6     296.3
7     230.1
8     163.5
9    232.84
Name: unit_price, dtype: object

In [97]:
Combine_Sales['unit_price'].map(type).value_counts()

unit_price
<class 'float'>    6625
<class 'str'>      3325
Name: count, dtype: int64

In [98]:
Combine_Sales['unit_price'] = pd.to_numeric(
    Combine_Sales['unit_price']
    .astype('string')
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip(),
    errors='coerce'
)

In [99]:
Combine_Sales['unit_price'].dtype

Float64Dtype()

# Extract specific promo codes from the sales notes field.

In [100]:
def clean_notes(notes):
    if pd.isna(notes) or notes == "":
        return pd.Series({'Promo Code': None, 'Discount Reason': None})
    
    promoCode = re.search(r'Promo code (\w+) applied at checkout', notes)
    
    if promoCode :
        return pd.Series({'Promo Code': promoCode.group(1), 'Discount Reason': 'Promo Code' })
    
    
    if 'loyalty' in notes.lower() :
        return pd.Series({'Promo Code': None, 'Discount Reason': 'Loyalty Tier Discount' })
    
    if 'competitor' in notes.lower() :
            return pd.Series({'Promo Code': None, 'Discount Reason': 'Price match' })
    
    return pd.Series({'Promo Code': None, 'Discount Reason': None})


Combine_Sales[['Promo Code', 'Discount Reason']] = Combine_Sales['notes'].apply(clean_notes)
Combine_Sales

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason
0,1,3,21,3878,99,2,2025-01-20,184.16,8.1,Cash,NaN,None,None
1,7,3,88,465,997,1,2024-10-20,108.23,6.9,Cash,NaN,None,None
2,8,2,32,3997,131,1,2024-07-12,298.25,6.4,Debit Card,NaN,None,None
3,12,2,13,4985,151,2,2025-07-26,140.55,5.4,Cash,NaN,None,None
4,13,1,3,3210,205,4,2025-09-03,179.13,32.4,Credit Card,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9945,7937,10,87,4504,372,3,2025-04-16,109.51,33.7,Cash,NaN,None,None
9946,6711,11,63,2028,805,1,2025-10-16,51.18,26.7,Credit Card,Loyalty tier discount applied,None,Loyalty Tier Discount
9947,8450,9,27,26,288,1,2025-04-10,48.36,5.2,Credit Card,Loyalty tier discount applied,None,Loyalty Tier Discount
9948,637,10,39,2571,282,1,2025-11-23,115.75,28.0,Credit Card,Promo code WELCOME15 applied at checkout,WELCOME15,Promo Code


In [101]:
Combine_Sales['Discount Reason'].value_counts(dropna=False)

Discount Reason
None                     8294
Promo Code                855
Price match               461
Loyalty Tier Discount     340
Name: count, dtype: int64

In [102]:
Combine_Sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9950 entries, 0 to 9949
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sale_id          9950 non-null   int64         
 1   store_id         9950 non-null   int64         
 2   employee_id      9950 non-null   int64         
 3   product_id       9950 non-null   int64         
 4   customer_id      9950 non-null   int64         
 5   quantity         9950 non-null   int64         
 6   sale_date        9950 non-null   datetime64[ns]
 7   unit_price       9950 non-null   Float64       
 8   discount_pct     9950 non-null   float64       
 9   payment_method   9950 non-null   object        
 10  notes            1656 non-null   object        
 11  Promo Code       855 non-null    object        
 12  Discount Reason  1656 non-null   object        
dtypes: Float64(1), datetime64[ns](1), float64(1), int64(6), object(4)
memory usage: 1020.4+ KB


In [103]:
Combine_Sales.shape

(9950, 13)

In [104]:
Combine_Sales.dtypes

sale_id                     int64
store_id                    int64
employee_id                 int64
product_id                  int64
customer_id                 int64
quantity                    int64
sale_date          datetime64[ns]
unit_price                Float64
discount_pct              float64
payment_method             object
notes                      object
Promo Code                 object
Discount Reason            object
dtype: object

In [105]:
nulls = pd.DataFrame({
    "null_count": Combine_Sales.isnull().sum(),
    "null_pct": (Combine_Sales.isnull().mean() * 100).round(2)
})

display(nulls[nulls["null_count"] > 0])

,null_count,null_pct
notes,8294,83.36
Promo Code,9095,91.41
Discount Reason,8294,83.36


notes, Promo Code, and Discount Reason contain expected nulls because one source sales file did not contain the notes field. These nulls represent unavailable source information and were retained.

# drop exact duplicate

In [106]:
display(Combine_Sales[Combine_Sales.duplicated(keep=False)])

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason
6645,222,12,108,3227,80,4,2025-01-24,95.93,3.6,Credit Card,NaN,None,None
6678,319,12,102,2076,2,3,2025-08-30,159.48,6.7,Cash,NaN,None,None
6684,332,11,81,3272,246,1,2025-09-01,96.34,31.5,Credit Card,NaN,None,None
6687,338,12,102,2938,371,5,2025-11-14,114.66,11.1,Cash,Price matched to competitor flyer,None,Price match
6720,424,10,71,2203,21,2,2025-04-25,24.29,31.2,Cash,Loyalty tier discount applied,None,Loyalty Tier Discount
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9945,7937,10,87,4504,372,3,2025-04-16,109.51,33.7,Cash,NaN,None,None
9946,6711,11,63,2028,805,1,2025-10-16,51.18,26.7,Credit Card,Loyalty tier discount applied,None,Loyalty Tier Discount
9947,8450,9,27,26,288,1,2025-04-10,48.36,5.2,Credit Card,Loyalty tier discount applied,None,Loyalty Tier Discount
9948,637,10,39,2571,282,1,2025-11-23,115.75,28.0,Credit Card,Promo code WELCOME15 applied at checkout,WELCOME15,Promo Code


In [107]:
duplicates = Combine_Sales[
    Combine_Sales.duplicated(keep=False)
].sort_values(by=Combine_Sales.columns.tolist())

display(duplicates)

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason
6645,222,12,108,3227,80,4,2025-01-24,95.93,3.6,Credit Card,NaN,None,None
9927,222,12,108,3227,80,4,2025-01-24,95.93,3.6,Credit Card,NaN,None,None
6678,319,12,102,2076,2,3,2025-08-30,159.48,6.7,Cash,NaN,None,None
9934,319,12,102,2076,2,3,2025-08-30,159.48,6.7,Cash,NaN,None,None
6684,332,11,81,3272,246,1,2025-09-01,96.34,31.5,Credit Card,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9941,9368,11,12,4566,196,1,2025-02-28,143.37,24.4,Debit Card,Promo code WELCOME15 applied at checkout,WELCOME15,Promo Code
9839,9739,9,61,4584,12,1,2025-09-07,44.51,11.1,Debit Card,Price matched to competitor flyer,None,Price match
9906,9739,9,61,4584,12,1,2025-09-07,44.51,11.1,Debit Card,Price matched to competitor flyer,None,Price match
9886,9859,10,33,2254,225,3,2025-08-26,157.14,30.8,Cash,NaN,None,None


In [108]:
duplicate_count = Combine_Sales.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 50


In [109]:
Combine_Sales = Combine_Sales.drop_duplicates().reset_index(drop=True)

In [145]:
duplicate_count = Combine_Sales.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


# check Unique Sales_id

In [146]:
key_cols = [
    'store_id',
    'employee_id',
    'product_id',
    'customer_id',
    'sale_date'
]

near_duplicates = Combine_Sales[
    Combine_Sales.duplicated(
        subset=key_cols,
        keep=False
    )
].sort_values(key_cols)

display(near_duplicates)

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason,cost_price,Revenue Before Discount,Revenue After Discount,Cost_Amount,profit,Margin_profit


In [110]:
display(
    Combine_Sales[
        Combine_Sales['sale_id'].duplicated(keep=False)
    ].sort_values('sale_id')
)

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason


In [112]:
numeric_cols = [
    "quantity",
    "unit_price",
    "discount_pct",
    "total_amount",
    "cost_amount",
    "profit",
    "margin_pct"
]

for col in numeric_cols:
    if col in Combine_Sales.columns:
        print(f"\n--- {col} ---")
        print("Min:", Combine_Sales[col].min())
        print("Max:", Combine_Sales[col].max())
        print("Negative values:", (Combine_Sales[col] < 0).sum())


--- quantity ---
Min: 1
Max: 5
Negative values: 0

--- unit_price ---
Min: 5.02
Max: 299.94
Negative values: 0

--- discount_pct ---
Min: 3.0
Max: 34.0
Negative values: 0


In [113]:
print(Combine_Sales["discount_pct"].describe())

count    9900.000000
mean       12.576293
std         9.268992
min         3.000000
25%         6.000000
50%         9.000000
75%        12.000000
max        34.000000
Name: discount_pct, dtype: float64


In [114]:
print("Discount > 1:", (Combine_Sales["discount_pct"] > 1).sum())
print("Discount > 0.50:", (Combine_Sales["discount_pct"] > 0.50).sum())
print("Discount < 0:", (Combine_Sales["discount_pct"] < 0).sum())

Discount > 1: 9900
Discount > 0.50: 9900
Discount < 0: 0


In [147]:
Combine_Sales['payment_method'] = (
    Combine_Sales['payment_method']
    .str.strip()
    .str.title()
)

In [148]:
(Combine_Sales['quantity'] < 0).sum()

0

In [150]:
(Combine_Sales['unit_price'] < 0).sum()

0

In [151]:
orphan_products = Combine_Sales[
    ~Combine_Sales['product_id'].isin(
        Products['product_id']
    )
]

print(
    "Product orphan rows:",
    len(orphan_products)
)

display(orphan_products)

Product orphan rows: 0


,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason,cost_price,Revenue Before Discount,Revenue After Discount,Cost_Amount,profit,Margin_profit


# Split pack_info into Pack Size, Strength, and Form.

In [115]:
Products = pd.read_csv(r"..\data\raw\products_catalog.csv")

In [116]:
Products['pack_info'].value_counts()

pack_info
Pack: 20g | Form: Cream                            116
Pack: 60g | Form: Cream                            116
Form: Device | Warranty: 12m                       110
Pack: 10g | Form: Cream                            107
Pack: 100g | Form: Cream                           104
                                                  ... 
Form: Liquid / Pack: 50ml / Strength: 25mg/5ml       1
Strength: 500mg/5ml / Pack: 450ml                    1
Form: Tablet / Pack: 20 tabs / Strength: 100mg       1
Pack: 100 tabs / Strength: 100mg / Form: Tablet      1
Form: Tablet / Pack: 10 tabs / Strength: 250mg       1
Name: count, Length: 626, dtype: int64

In [117]:
Products['Pack'] = Products['pack_info'].str.extract(
    r'Pack:\s*([^|/]+)',
    expand=False
)

Products['Strength'] = Products['pack_info'].str.extract(
       r'Strength:\s*(.*?)(?=\s*(?:\||/)\s*(?:Pack|Form|Warranty|Name):|$)',
    expand=False
)

Products['Form'] = Products['pack_info'].str.extract(
    r'Form:\s*(.*?)(?=\s*(?:\||/)\s*(?:Pack|Strength|Warranty|Name):|$)',
    expand=False
)


Products['Pack'] = Products['Pack'].str.strip()
Products['Strength'] = Products['Strength'].str.strip()
Products['Form'] = Products['Form'].str.strip()


In [118]:
Products

,product_id,product_name,category_id,supplier_id,unit_price,cost_price,brand,expiry_date,storage_condition,created_at,pack_info,Pack,Strength,Form
0,1,Health_Product_1,4,2,98.97,107.40,BioZen,2026-11-29,Refrigerated,2022-03-12,Pack: 10g | Form: Cream,10g,NaN,Cream
1,2,Health_Product_2,8,3,111.42,64.75,MediCare,2026-08-08,Refrigerated,2022-09-13,NaN,NaN,NaN,NaN
2,3,Health_Product_3,9,4,14.86,7.49,CuraMax,2027-03-15,Refrigerated,2023-06-25,Pack: 10 units | Form: Kit,10 units,NaN,Kit
3,4,Health_Product_4,5,4,274.78,153.13,VitalMeds,2029-09-28,Controlled,2022-09-26,Form: Device | Warranty: 12m,NaN,NaN,Device
4,5,Health_Product_5,2,3,87.09,109.66,CuraMax,2027-01-05,Controlled,2025-10-07,Pack: 10 tabs | Strength: 250mg | Form: Tablet,10 tabs,250mg,Tablet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,Health_Product_4996,5,3,251.19,176.95,BioZen,2026-07-15,Ambient,2023-12-11,Warranty: 6m / Form: Device,NaN,NaN,Device
4996,4997,Health_Product_4997,9,1,111.39,70.71,PharmaCo,2028-08-04,Controlled,2024-11-26,Form: Kit / Pack: 60 units,60 units,NaN,Kit
4997,4998,Health_Product_4998,7,1,292.62,167.81,CuraMax,2026-05-27,Refrigerated,2023-04-06,Pack: 90g | Form: Bandage,90g,NaN,Bandage
4998,4999,Health_Product_4999,5,3,251.79,180.42,BioZen,2026-11-30,Refrigerated,2025-03-07,Form: Device | Warranty: 24m,NaN,NaN,Device


In [119]:
# !pip install ydata_profiling --quiet

In [120]:
# from ydata_profiling import ProfileReport

In [121]:
# profile = ProfileReport(Sales , title = "Sales Data Profiling Report")
# profile.to_file("Sales Data Profiling Report.html")
# profile

# Join sales data with the product catalog's cost_price. Compute derived financial metrics including cost_amount, profit, and margin_pct.

In [122]:
Combine_Sales = Combine_Sales.merge(Products[['product_id','cost_price']])

In [123]:
Combine_Sales

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason,cost_price
0,1,3,21,3878,99,2,2025-01-20,184.16,8.1,Cash,NaN,None,None,81.69
1,7,3,88,465,997,1,2024-10-20,108.23,6.9,Cash,NaN,None,None,57.85
2,8,2,32,3997,131,1,2024-07-12,298.25,6.4,Debit Card,NaN,None,None,212.63
3,12,2,13,4985,151,2,2025-07-26,140.55,5.4,Cash,NaN,None,None,90.61
4,13,1,3,3210,205,4,2025-09-03,179.13,32.4,Credit Card,NaN,None,None,120.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9895,9882,10,15,3811,102,5,2025-02-22,281.85,26.3,Online,Price matched to competitor flyer,None,Price match,120.74
9896,9884,10,15,3636,292,5,2024-11-04,159.76,23.6,Cash,Promo code LOYALTY20 applied at checkout,LOYALTY20,Promo Code,95.80
9897,9885,11,81,430,808,5,2024-12-26,141.68,33.4,Credit Card,NaN,None,None,67.60
9898,9888,9,24,968,1023,1,2025-09-06,230.71,11.1,Credit Card,Loyalty tier discount applied,None,Loyalty Tier Discount,145.52


In [124]:
Combine_Sales['Revenue Before Discount'] =  Combine_Sales['quantity'] * Combine_Sales['unit_price'] 

In [125]:
Combine_Sales['Revenue After Discount'] = (Combine_Sales['Revenue Before Discount'] - ( (Combine_Sales['Revenue Before Discount'] *( Combine_Sales['discount_pct']/100))))

In [126]:
Combine_Sales['Cost_Amount'] = Combine_Sales['quantity'] * Combine_Sales['cost_price']

In [127]:
Combine_Sales['profit'] = Combine_Sales['Revenue After Discount'] - Combine_Sales['Cost_Amount']

In [128]:
Combine_Sales['Margin_profit'] = (Combine_Sales['profit'] / Combine_Sales['Revenue After Discount']) * 100

In [129]:
Combine_Sales

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,Promo Code,Discount Reason,cost_price,Revenue Before Discount,Revenue After Discount,Cost_Amount,profit,Margin_profit
0,1,3,21,3878,99,2,2025-01-20,184.16,8.1,Cash,NaN,None,None,81.69,368.32,338.48608,163.38,175.10608,51.732136
1,7,3,88,465,997,1,2024-10-20,108.23,6.9,Cash,NaN,None,None,57.85,108.23,100.76213,57.85,42.91213,42.587557
2,8,2,32,3997,131,1,2024-07-12,298.25,6.4,Debit Card,NaN,None,None,212.63,298.25,279.162,212.63,66.532,23.832757
3,12,2,13,4985,151,2,2025-07-26,140.55,5.4,Cash,NaN,None,None,90.61,281.1,265.9206,181.22,84.7006,31.851838
4,13,1,3,3210,205,4,2025-09-03,179.13,32.4,Credit Card,NaN,None,None,120.41,716.52,484.36752,481.64,2.72752,0.56311
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9895,9882,10,15,3811,102,5,2025-02-22,281.85,26.3,Online,Price matched to competitor flyer,None,Price match,120.74,1409.25,1038.61725,603.70,434.91725,41.874642
9896,9884,10,15,3636,292,5,2024-11-04,159.76,23.6,Cash,Promo code LOYALTY20 applied at checkout,LOYALTY20,Promo Code,95.80,798.8,610.2832,479.00,131.2832,21.511849
9897,9885,11,81,430,808,5,2024-12-26,141.68,33.4,Credit Card,NaN,None,None,67.60,708.4,471.7944,338.00,133.7944,28.358624
9898,9888,9,24,968,1023,1,2025-09-06,230.71,11.1,Credit Card,Loyalty tier discount applied,None,Loyalty Tier Discount,145.52,230.71,205.10119,145.52,59.58119,29.049656


# VALIDATION 1: Revenue Before Discount

In [152]:
expected_revenue_before = (
    Combine_Sales['quantity'] *
    Combine_Sales['unit_price']
)

revenue_before_errors = (
    ~np.isclose(
        Combine_Sales['Revenue Before Discount'],
        expected_revenue_before,
        rtol=1e-05,
        atol=1e-08
    )
)

print(
    "Revenue Before Discount errors:",
    revenue_before_errors.sum()
)

Revenue Before Discount errors: 0


# VALIDATION 2: REVENUE AFTER DISCOUNT

In [153]:
expected_revenue_after = (
    Combine_Sales['Revenue Before Discount']
    * (1 - Combine_Sales['discount_pct'] / 100)
)

revenue_after_errors = (
    ~np.isclose(
        Combine_Sales['Revenue After Discount'],
        expected_revenue_after,
        rtol=1e-05,
        atol=1e-08
    )
)

print(
    "Revenue After Discount errors:",
    revenue_after_errors.sum()
)

Revenue After Discount errors: 0


# VALIDATION 3: COST AMOUNT

In [154]:
expected_cost = (
    Combine_Sales['quantity'] *
    Combine_Sales['cost_price']
)

cost_errors = (
    ~np.isclose(
        Combine_Sales['Cost_Amount'],
        expected_cost,
        rtol=1e-05,
        atol=1e-08
    )
)

print(
    "Cost Amount errors:",
    cost_errors.sum()
)

Cost Amount errors: 0


# VALIDATION 4: PROFIT

In [155]:
expected_profit = (
    Combine_Sales['Revenue After Discount']
    - Combine_Sales['Cost_Amount']
)

profit_errors = (
    ~np.isclose(
        Combine_Sales['profit'],
        expected_profit,
        rtol=1e-05,
        atol=1e-08
    )
)

print(
    "Profit errors:",
    profit_errors.sum()
)

Profit errors: 0


# VALIDATION 5: MARGIN %

In [156]:
expected_margin = (
    Combine_Sales['profit']
    / Combine_Sales['Revenue After Discount']
) * 100

margin_errors = (
    ~np.isclose(
        Combine_Sales['Margin_profit'],
        expected_margin,
        rtol=1e-05,
        atol=1e-08
    )
)

print(
    "Margin % errors:",
    margin_errors.sum()
)

Margin % errors: 0


# FK VALIDATION 1: SALES → PRODUCTS

In [157]:
product_orphans = Combine_Sales[
    ~Combine_Sales['product_id'].isin(
        Products['product_id']
    )
]

print(
    "Sales rows with invalid product_id:",
    len(product_orphans)
)

Sales rows with invalid product_id: 0


# FK VALIDATION 2: SALES → STORES

In [159]:
store_orphans = Combine_Sales[
    ~Combine_Sales['store_id'].isin(
        Stores['store_id']
    )
]

print(
    "Sales rows with invalid store_id:",
    len(store_orphans)
)

Sales rows with invalid store_id: 0


# FK VALIDATION 3: SALES → EMPLOYEES

In [165]:
employee_orphans = Combine_Sales[
    ~Combine_Sales['employee_id'].isin(
        Employee['employee_id']
    )
]

print(
    "Sales rows with invalid employee_id:",
    len(employee_orphans)
)

Sales rows with invalid employee_id: 0


# FK VALIDATION 4: SALES → CUSTOMERS

In [166]:
customer_orphans = Combine_Sales[
    ~Combine_Sales['customer_id'].isin(
        customers['customer_id']
    )
]

print(
    "Sales rows with invalid customer_id:",
    len(customer_orphans)
)

Sales rows with invalid customer_id: 0


# BUSINESS RULE 1: MARGIN VALIDATION

In [167]:
negative_margin = Combine_Sales[
    Combine_Sales['Margin_profit'] < 0
]

high_margin = Combine_Sales[
    Combine_Sales['Margin_profit'] > 80
]

print(
    "Negative margin rows:",
    len(negative_margin)
)

print(
    "Margin above 80% rows:",
    len(high_margin)
)

Negative margin rows: 608
Margin above 80% rows: 0


In [168]:
display(
    negative_margin[
        [
            'sale_id',
            'product_id',
            'quantity',
            'unit_price',
            'discount_pct',
            'cost_price',
            'Revenue After Discount',
            'Cost_Amount',
            'profit',
            'Margin_profit'
        ]
    ].head(20)
)

,sale_id,product_id,quantity,unit_price,discount_pct,cost_price,Revenue After Discount,Cost_Amount,profit,Margin_profit
68,188,4272,5,48.45,5.0,61.06,230.1375,305.30,-75.1625,-32.659823
94,263,3824,2,240.16,8.9,269.73,437.57152,539.46,-101.88848,-23.284989
106,307,2525,1,79.54,7.3,99.96,73.73358,99.96,-26.22642,-35.569167
122,360,4939,1,34.17,5.3,40.41,32.35899,40.41,-8.05101,-24.880288
136,427,4906,4,21.02,10.1,26.84,75.58792,107.36,-31.77208,-42.033277
154,479,3283,4,218.74,32.2,150.34,593.22288,601.36,-8.13712,-1.37168
159,504,4602,5,191.63,3.9,241.95,920.78215,1209.75,-288.96785,-31.382868
199,639,1694,1,195.16,23.1,217.95,150.07804,217.95,-67.87196,-45.224445
202,649,4085,2,196.68,31.3,139.42,270.23832,278.84,-8.60168,-3.182998
220,717,1060,1,22.69,30.3,16.03,15.81493,16.03,-0.21507,-1.359917


# BUSINESS RULE 2: DISCOUNT VALIDATION

In [169]:
negative_discount = Combine_Sales[
    Combine_Sales['discount_pct'] < 0
]

above_discount_cap = Combine_Sales[
    Combine_Sales['discount_pct'] > 50
]

print(
    "Negative discount rows:",
    len(negative_discount)
)

print(
    "Discount above 50% rows:",
    len(above_discount_cap)
)

Negative discount rows: 0
Discount above 50% rows: 0


# BUSINESS RULE 3: QUANTITY VALIDATION

In [170]:
invalid_quantity = Combine_Sales[
    Combine_Sales['quantity'] <= 0
]

print(
    "Invalid quantity rows:",
    len(invalid_quantity)
)

Invalid quantity rows: 0


# BUSINESS RULE 4: PRICE VALIDATION

In [171]:
invalid_unit_price = Combine_Sales[
    Combine_Sales['unit_price'] <= 0
]

print(
    "Invalid unit price rows:",
    len(invalid_unit_price)
)

Invalid unit price rows: 0


# BUSINESS RULE 5: COST VALIDATION

In [172]:
invalid_cost_price = Combine_Sales[
    Combine_Sales['cost_price'] <= 0
]

print(
    "Invalid cost price rows:",
    len(invalid_cost_price)
)

Invalid cost price rows: 0


# BUSINESS RULE 6: DATE VALIDATION

In [173]:
invalid_dates = Combine_Sales[
    Combine_Sales['sale_date'].isna()
]

print(
    "Invalid / missing sale dates:",
    len(invalid_dates)
)

Invalid / missing sale dates: 0


# VALIDATION REPORT

In [174]:
validation_results = []

def add_validation_result(rule, count, severity):
    validation_results.append({
        'table': 'fact_sales',
        'rule_violated': rule,
        'row_count': int(count),
        'severity': severity
    })

In [175]:
add_validation_result(
    'Revenue Before Discount calculation error',
    revenue_before_errors.sum(),
    'ERROR'
)

add_validation_result(
    'Revenue After Discount calculation error',
    revenue_after_errors.sum(),
    'ERROR'
)

add_validation_result(
    'Cost Amount calculation error',
    cost_errors.sum(),
    'ERROR'
)

add_validation_result(
    'Profit calculation error',
    profit_errors.sum(),
    'ERROR'
)

add_validation_result(
    'Margin calculation error',
    margin_errors.sum(),
    'ERROR'
)

In [176]:
add_validation_result(
    'Invalid product_id',
    len(product_orphans),
    'ERROR'
)

add_validation_result(
    'Invalid product_id',
    len(product_orphans),
    'ERROR'
)

add_validation_result(
    'Invalid employee_id',
    len(employee_orphans),
    'ERROR'
)

add_validation_result(
    'Invalid customer_id',
    len(customer_orphans),
    'ERROR'
)

In [177]:
add_validation_result(
    'Negative quantity',
    len(invalid_quantity),
    'ERROR'
)

add_validation_result(
    'Negative unit price',
    len(invalid_unit_price),
    'ERROR'
)

add_validation_result(
    'Missing / invalid sale date',
    len(invalid_dates),
    'ERROR'
)

add_validation_result(
    'Negative discount',
    len(negative_discount),
    'ERROR'
)

add_validation_result(
    'Discount above 50% policy ceiling',
    len(above_discount_cap),
    'WARN'
)

add_validation_result(
    'Negative margin',
    len(negative_margin),
    'WARN'
)

add_validation_result(
    'Margin above 80%',
    len(high_margin),
    'WARN'
)

add_validation_result(
    'Invalid cost price',
    len(invalid_cost_price),
    'ERROR'
)

In [178]:
validation_report = pd.DataFrame(validation_results)

display(validation_report)

,table,rule_violated,row_count,severity
0,fact_sales,Revenue Before Discount calculation error,0,ERROR
1,fact_sales,Revenue After Discount calculation error,0,ERROR
2,fact_sales,Cost Amount calculation error,0,ERROR
3,fact_sales,Profit calculation error,0,ERROR
4,fact_sales,Margin calculation error,0,ERROR
5,fact_sales,Invalid product_id,0,ERROR
6,fact_sales,Invalid product_id,0,ERROR
7,fact_sales,Invalid employee_id,0,ERROR
8,fact_sales,Invalid customer_id,0,ERROR
9,fact_sales,Negative quantity,0,ERROR


# FINAL VALIDATION STATUS

In [179]:
errors = validation_report[
    validation_report['severity'] == 'ERROR'
]

warnings = validation_report[
    validation_report['severity'] == 'WARN'
]

print("Total ERROR rows across checks:", errors['row_count'].sum())
print("Total WARN rows across checks:", warnings['row_count'].sum())

if errors['row_count'].sum() == 0:
    print("✅ SALES VALIDATION PASSED")
else:
    print("❌ SALES VALIDATION FAILED - Fix ERRORs before export")

Total ERROR rows across checks: 0
Total WARN rows across checks: 608
✅ SALES VALIDATION PASSED


In [180]:
validation_report.to_csv(
    '../validation/validation_sales_report.csv',
    index=False
)

# Build extraction logic for embedded customer details (such as loyalty status) and apply it to the full customer file.

In [131]:
customers = pd.read_csv(r"..\data\raw\customers_master.csv")

In [132]:
def extract_loyalty_status(notes):
    if pd.isna(notes) or notes == "":
        return pd.Series({'loyalty_status': None , 'loyalty_since': None})
    
    loyalty_status = re.search(r'(Gold|Silver|Platinum) member - loyalty since (\d{4})' , notes)
    
    if loyalty_status:
        return pd.Series({'loyalty_status': loyalty_status.group(1) , 'loyalty_since': loyalty_status.group(2)})
    
    return pd.Series({'loyalty_status': None , 'loyalty_since': None})

loyalty_status = customers['notes'].apply(extract_loyalty_status)
customers = pd.concat([customers , loyalty_status], axis=1)
customers

,customer_id,full_name,email,phone,city,signup_date,notes,loyalty_status,loyalty_since
0,269,Mary Nguyen,mary.nguyen269@yahoo.com,8728923567,Montreal,2022-01-15,Prefers text reminders,None,None
1,578,John Campbell,john_campbell578@hotmail.com,4659426815,Montreal,2019-05-28,NaN,None,None
2,482,Joseph Roy,joseph.roy482@hotmail.com,921.818.9302,Edmonton,2020-12-23,Prefers text reminders,None,None
3,1501,Nancy Taaylor,nancytaaylor1501@outlook.com,7821678948,Montreal,2023-02-12,NaN,None,None
4,1506,Kevin Halll,kevinhalll1506@yahoo.com,(763) 054-4575,Moncton,2025-07-13,Opted out of email marketing,None,None
...,...,...,...,...,...,...,...,...,...
1550,716,Justin Campbell,justin.campbell716@gmail.com,(658) 174-1647,Montreal,2024-06-30,NaN,None,None
1551,906,Matthew Tremblay,matthewtremblay906@hotmail.com,(488) 857-0064,Quebec City,2021-05-10,Platinum member - loyalty since 2019,Platinum,2019
1552,1097,Kevin Nguyen,kevinnguyen1097@hotmail.com,(533) 247-3674,Kitchener,2023-06-08,NaN,None,None
1553,236,Amanda Campbell,amanda_campbell236@yahoo.com,5493917209,Moncton,2022-05-23,NaN,None,None


In [133]:
customers['loyalty_status'].value_counts(dropna=False)

loyalty_status
None        1099
Silver       158
Platinum     154
Gold         144
Name: count, dtype: int64

# Handle typo'd names, reformatted phone numbers, and duplicate entries for the same person.

In [134]:
customers['Clean_phone'] = customers['phone'].astype('string').str.replace(r'\D','', regex=True)

In [135]:
customers['city'] = customers['city'].astype('string').str.strip().str.title()

In [136]:
duplicate_phones = []

for phone , ids in customers.groupby('Clean_phone'):
    if len(ids) > 1:
        duplicate_phones.append(phone)

print(duplicate_phones)

['0120312738', '0346718657', '0903958146', '0940211340', '1070983552', '1232566042', '1387506072', '1403780254', '1918453122', '2082302039', '2364021542', '2519836473', '2719200342', '2861539636', '2930495031', '3070107286', '3100486159', '3122820605', '3350119341', '3914256511', '3991931598', '4453111141', '4554728484', '5156393203', '5213537876', '5560399891', '6091391101', '6296743145', '6474935237', '6480297227', '6592169945', '6619577471', '6819081876', '7110878738', '7508920694', '7624741910', '7630544575', '7821678948', '8408967147', '8982484003', '9091405791', '9507472950']


In [137]:
customers

,customer_id,full_name,email,phone,city,signup_date,notes,loyalty_status,loyalty_since,Clean_phone
0,269,Mary Nguyen,mary.nguyen269@yahoo.com,8728923567,Montreal,2022-01-15,Prefers text reminders,None,None,8728923567
1,578,John Campbell,john_campbell578@hotmail.com,4659426815,Montreal,2019-05-28,NaN,None,None,4659426815
2,482,Joseph Roy,joseph.roy482@hotmail.com,921.818.9302,Edmonton,2020-12-23,Prefers text reminders,None,None,9218189302
3,1501,Nancy Taaylor,nancytaaylor1501@outlook.com,7821678948,Montreal,2023-02-12,NaN,None,None,7821678948
4,1506,Kevin Halll,kevinhalll1506@yahoo.com,(763) 054-4575,Moncton,2025-07-13,Opted out of email marketing,None,None,7630544575
...,...,...,...,...,...,...,...,...,...,...
1550,716,Justin Campbell,justin.campbell716@gmail.com,(658) 174-1647,Montreal,2024-06-30,NaN,None,None,6581741647
1551,906,Matthew Tremblay,matthewtremblay906@hotmail.com,(488) 857-0064,Quebec City,2021-05-10,Platinum member - loyalty since 2019,Platinum,2019,4888570064
1552,1097,Kevin Nguyen,kevinnguyen1097@hotmail.com,(533) 247-3674,Kitchener,2023-06-08,NaN,None,None,5332473674
1553,236,Amanda Campbell,amanda_campbell236@yahoo.com,5493917209,Moncton,2022-05-23,NaN,None,None,5493917209


# Normalize doctor-name casing and dosage abbreviations.

In [138]:
prescriptions = pd.read_csv(r"..\data\raw\prescriptions_log.csv")

In [139]:
prescriptions['doctor_name'] = prescriptions['doctor_name'].str.strip().str.title()
prescriptions['doctor_name'] = prescriptions['doctor_name'].str.replace(r'^Dr\.?\s*', 'Dr. ' , regex=True)

In [140]:
prescriptions

,prescription_id,store_id,product_id,customer_id,doctor_name,issue_date,dosage
0,1,9,3881,120,Dr. Chen,2025-10-17,PRN
1,2,8,4828,419,Dr. Chen,2024-10-20,1x daily
2,3,1,4828,867,Dr. Singh,2025-10-15,Inject once weekly
3,4,2,880,153,Dr. Garcia,2025-11-24,as needed
4,5,8,2523,97,Dr. King,2025-06-09,as needed
...,...,...,...,...,...,...,...
9995,9996,2,2451,252,Dr. Jones,2024-11-01,BID
9996,9997,5,3430,894,Dr. Garcia,2025-10-03,2x daily
9997,9998,8,2785,1129,Dr. King,2025-01-29,1x daily
9998,9999,6,4664,733,Dr. King,2024-12-28,1x daily


In [141]:
prescriptions['dosage'].value_counts()

dosage
Inject once weekly    745
1 tab/day             714
1 tablet daily        695
PRN                   688
1 injection weekly    686
5ml q6h               684
BID                   679
Twice a day           656
As needed             653
as needed             645
5ml every 6 hours     644
1x daily              642
inj 1x/wk             629
2x daily              623
5 ml every 6 hrs      617
Name: count, dtype: int64

In [142]:
dosage_mapping = {
    'Inject once weekly': 'Once weekly',
    '1 tab/day': 'Once daily',
    '1 tablet daily': 'Once daily',
    'PRN': 'As needed',
    '1 injection weekly': 'Once weekly',
    '5ml q6h': '5ml Every 6 hours',
    'BID': 'Twice daily',
    'Twice a day': 'Twice daily',
    'As needed': 'As needed',
    'as needed': 'As needed',
    '5ml every 6 hours': '5ml Every 6 hours',
    '1x daily': 'Once daily',
    'inj 1x/wk': 'Once weekly',
    '2x daily': 'Twice daily',
    '5 ml every 6 hrs': '5ml Every 6 hours'
}

prescriptions['dosage'] = prescriptions['dosage'].replace(dosage_mapping)

In [143]:
prescriptions['dosage'].value_counts()

dosage
Once weekly          2060
Once daily           2051
As needed            1986
Twice daily          1958
5ml Every 6 hours    1945
Name: count, dtype: int64

# Saving Cleaned CSV files

In [144]:
Combine_Sales.to_csv(r'..\data\cleaned\Combine_Sales.csv', index=False)

PermissionError: [Errno 13] Permission denied: '..\\data\\cleaned\\Combine_Sales.csv'

In [ ]:
Products.to_csv(r'..\data\cleaned\Products.csv', index=False)

In [ ]:
prescriptions.to_csv(r'..\data\cleaned\prescriptions.csv' , index=False)

In [ ]:
#pd.set_option('display.max_rows',9950)
pd.reset_option('display.max_rows')